# Logit lens: Qwen3-30B vs Qwen3-4B — DAP에서 대답이 어디서 갈라지는가

**질문**: 같은 그룹 C 문항, 같은 DAP 프롬프트에서 Qwen3-30B는 '거짓'(미완료 과신), Qwen3-4B는 '모름'(정답)을 고른다.
두 모델의 **층별로 참/거짓/모름 확률이 어떻게 변하는지**를 비교해, 30B의 답이 어느 깊이에서 '거짓' 쪽으로 기우는지 확인한다.

**단계** (위에서부터 순서대로 실행)
- **0단계 — 환경 확인**: GPU 2개, 디스크, 메모리
- **1단계 — 4B**: 행동 재확인(그룹 C × Zero-shot/DAP, 200회) + logit lens
- **2단계 — 30B 재현 확인**: Kaggle 환경(bnb 4비트)에서도 DAP '거짓' 급증이 재현되는지 사전 기준으로 판정
  - 기준(대리 모델 때와 동일, 결과를 보기 전에 고정): FBR_C(DAP) − FBR_C(Zero-shot) ≥ 0.15 **그리고** 정확 McNemar p < 0.05
- **3단계 — 30B logit lens** (2단계에서 재현됐을 때만 해석의 중심으로 사용)
- **4단계 — 비교 그림과 결과 저장**

**Kaggle 설정 (실행 전에 꼭)**
1. 오른쪽 패널 **Session options → Accelerator → GPU T4 x2**
2. **Internet → On** (휴대폰 인증된 계정만 켜짐)
3. 전체 실행 예상: 4B 약 15분 + 30B 다운로드·로드 20~30분 + 30B 실행 약 30분. 결과는 `/kaggle/working/lens_out/`에 저장되고, 세션이 끊겨도 끝난 부분은 건너뛴다.

**본실험과의 차이 (논문 방법론에 적을 것)**: 본실험의 30B는 Ollama q4_K_M(GGUF), 여기서는 transformers + bitsandbytes NF4. 그래서 2단계에서 행동이 같은지 먼저 확인한다.

## 0. 설정

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # GPU 메모리 조각남 방지 (torch를 불러오기 전에 설정)
RUN_4B = True
RUN_30B = True
MODEL_4B = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_30B = "Qwen/Qwen3-30B-A3B-Instruct-2507"
MODEL_30B_LOCAL = None      # Kaggle 'Add Input'으로 30B 가중치를 붙였다면 그 경로 (예: '/kaggle/input/...'), 아니면 None
OUT_DIR = "/kaggle/working/lens_out"
STRATEGIES = ["zero-shot", "dap"]
MAX_NEW_TOKENS = 16

## 0-1. 패키지 (transformers 버전 고정이 중요)
5.x에서는 MoE 전문가 층 구조가 바뀌어 bitsandbytes 4비트 변환이 전문가 층에 적용되지 않을 수 있어, 구조가 검증된 4.57.1로 고정한다.

Kaggle에 미리 깔린 패키지 중 **torchvision·torchaudio·kernels**는 transformers 5.x에 맞춰져 있어 4.57.1과 충돌할 수 있다(모델 코드 불러오기가 실패하면 transformers는 진짜 원인을 숨기고 'Could not find Qwen3ForCausalLM'만 보여준다). 텍스트 모델 실험에는 필요 없으므로 먼저 제거한다.

⚠️ **이 셀을 처음 실행한 뒤에는 반드시 메뉴 Run → Restart & clear cell outputs를 누르고, 이 셀부터 다시 실행**한다. (이미 받은 모델 파일은 지워지지 않는다.)

In [ ]:
!pip uninstall -y -q torchvision torchaudio kernels
!pip install -q "transformers==4.57.1" "accelerate>=1.0,<2" "bitsandbytes>=0.45" "huggingface_hub>=0.34,<1.0"
import importlib, traceback, transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)
assert transformers.__version__ == '4.57.1', '버전이 다릅니다 → Run → Restart & clear cell outputs 후 이 셀부터 다시 실행'
# 모델 코드를 직접 불러와서, 실패하면 숨겨진 진짜 에러를 그대로 보여준다
for mod in ['transformers.models.qwen3.modeling_qwen3', 'transformers.models.qwen3_moe.modeling_qwen3_moe']:
    try:
        importlib.import_module(mod); print('OK', mod)
    except Exception:
        traceback.print_exc()
        raise SystemExit('위 에러 전체를 복사해서 Claude에게 보내주세요.')

## 0-2. 환경 확인

In [ ]:
import os, shutil, json, gc, time
import psutil
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i); print(f'GPU{i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')
cands = ['/kaggle/tmp', '/tmp', '/root', '/kaggle/working']
free = {}
for d in cands:
    try:
        os.makedirs(d, exist_ok=True); free[d] = shutil.disk_usage(d).free/1e9
    except Exception as e:
        pass
for d, f in free.items(): print(f'디스크 여유 {d}: {f:.0f} GB')
CACHE_DIR = max(free, key=free.get)
REPO_30B_DIR = "models--" + MODEL_30B.replace("/", "--")
for d in cands:   # 이미 30B를 받아 둔 폴더가 있으면 그곳을 그대로 사용
    if os.path.isdir(os.path.join(d, REPO_30B_DIR)): CACHE_DIR = d
print('모델 저장 위치 →', CACHE_DIR)
if RUN_30B and MODEL_30B_LOCAL is None and free[CACHE_DIR] < 66 and not os.path.isdir(os.path.join(CACHE_DIR, REPO_30B_DIR)):
    print('\n⚠️ 30B 원본(약 61GB)을 받을 디스크가 부족합니다. 4B는 그대로 진행되고, 30B는 건너뜁니다.')
    print('   대안: Kaggle "Add Input"에서 Qwen3-30B-A3B-Instruct-2507 가중치를 붙이고 MODEL_30B_LOCAL에 경로 지정')
    RUN_30B = False
assert n_gpu >= 1, 'GPU가 없습니다. Session options에서 GPU T4 x2를 선택하세요.'
if RUN_30B: assert n_gpu >= 2, '30B는 GPU 2개가 필요합니다 (GPU T4 x2).'
os.makedirs(OUT_DIR, exist_ok=True)

## 0-3. 데이터 (그룹 C 100문항) · 프롬프트 · 파서 — 본실험과 동일

In [ ]:
ITEMS_C = json.loads(r'''[{"id": "C_001", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "고치다", "premise": "그는 고장 난 시계를 고치고 있었다.", "hypothesis": "그는 고장 난 시계를 다 고쳤다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_002", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "채우다", "premise": "그녀는 컵에 물을 채우고 있었다.", "hypothesis": "그녀는 컵에 물을 가득 채웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_003", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "청소하다", "premise": "그는 방 하나를 청소하고 있었다.", "hypothesis": "그는 방 하나 전체를 청소했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_004", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "수리하다", "premise": "정비사가 차 한 대를 수리하고 있었다.", "hypothesis": "정비사가 차 한 대를 다 수리했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_005", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "비우다", "premise": "그는 서랍을 비우고 있었다.", "hypothesis": "그는 서랍을 다 비웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_006", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "말리다", "premise": "그녀는 빨래를 말리고 있었다.", "hypothesis": "그녀는 빨래를 완전히 말렸다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_007", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "세탁하다", "premise": "그는 옷 한 벌을 세탁하고 있었다.", "hypothesis": "그는 옷 한 벌을 다 세탁했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_008", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "다림질하다", "premise": "그녀는 셔츠 한 장을 다림질하고 있었다.", "hypothesis": "그녀는 셔츠 한 장을 다 다림질했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_009", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "깎다", "premise": "그는 연필을 깎고 있었다.", "hypothesis": "그는 연필을 다 깎았다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_010", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "부풀리다", "premise": "그는 풍선을 최대한 부풀리고 있었다.", "hypothesis": "그는 풍선을 최대한 다 부풀렸다.", "ollama30b_zs": "True", "ollama30b_dap": "False"}, {"id": "C_011", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "얼리다", "premise": "그녀는 물을 얼리고 있었다.", "hypothesis": "그녀는 물을 다 얼렸다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_012", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "녹이다", "premise": "그는 버터를 녹이고 있었다.", "hypothesis": "그는 버터를 다 녹였다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_013", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "봉합하다", "premise": "의사가 상처를 봉합하고 있었다.", "hypothesis": "의사가 상처를 다 봉합했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_014", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "개조하다", "premise": "그들은 집을 개조하고 있었다.", "hypothesis": "그들은 집을 다 개조했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_015", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "철거하다", "premise": "인부들이 건물을 철거하고 있었다.", "hypothesis": "인부들이 건물을 다 철거했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_016", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "파다", "premise": "그는 구덩이 하나를 파고 있었다.", "hypothesis": "그는 구덩이 하나를 다 팠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_017", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "싸다", "premise": "그녀는 짐을 싸고 있었다.", "hypothesis": "그녀는 짐을 다 쌌다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_018", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "풀다", "premise": "그는 매듭 하나를 풀고 있었다.", "hypothesis": "그는 매듭 하나를 다 풀었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_019", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "펴다", "premise": "그녀는 구겨진 종이를 펴고 있었다.", "hypothesis": "그녀는 구겨진 종이를 다 폈다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_020", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "해독하다", "premise": "그는 암호 하나를 해독하고 있었다.", "hypothesis": "그는 암호 하나를 다 해독했다.", "ollama30b_zs": "False", "ollama30b_dap": "False"}, {"id": "C_021", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "정리하다", "premise": "그는 책상 하나를 정리하고 있었다.", "hypothesis": "그는 책상 하나를 다 정리했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_022", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "지우다", "premise": "그녀는 칠판을 지우고 있었다.", "hypothesis": "그녀는 칠판을 전부 지웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_023", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "수정하다", "premise": "그는 보고서를 수정하고 있었다.", "hypothesis": "그는 보고서를 다 수정했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_024", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "설치하다", "premise": "그녀는 프로그램을 설치하고 있었다.", "hypothesis": "그녀는 프로그램을 다 설치했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_025", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "포장하다", "premise": "그는 선물을 포장하고 있었다.", "hypothesis": "그는 선물을 다 포장했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_026", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "제거하다", "premise": "그녀는 자전거의 녹을 제거하고 있었다.", "hypothesis": "그녀는 자전거의 녹을 다 제거했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_027", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "메우다", "premise": "그는 벽의 틈을 메우고 있었다.", "hypothesis": "그는 벽의 틈을 다 메웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_028", "verb_class": "Change_of_State", "secondary_class": "전형적 달성동사", "verb": "구조하다", "premise": "소방관이 아이를 구조하고 있었다.", "hypothesis": "소방관이 아이를 다 구조했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_029", "verb_class": "Change_of_State", "secondary_class": "전형적 달성동사", "verb": "구해내다", "premise": "그는 물에 빠진 사람을 구해내고 있었다.", "hypothesis": "그는 물에 빠진 사람을 다 구해냈다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_030", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "다듬다", "premise": "그는 잔디밭을 다듬고 있었다.", "hypothesis": "그는 잔디밭 전체를 다듬었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_031", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "쓸다", "premise": "그녀는 바닥을 쓸고 있었다.", "hypothesis": "그녀는 바닥 전체를 쓸었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_032", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "닦다", "premise": "그는 복도를 대걸레로 닦고 있었다.", "hypothesis": "그는 복도 전체를 대걸레로 닦았다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_033", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "광내다", "premise": "그녀는 구두를 광내고 있었다.", "hypothesis": "그녀는 구두를 다 광냈다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_034", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "사포질하다", "premise": "그는 책상을 사포질하고 있었다.", "hypothesis": "그는 책상 전체를 사포질했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_035", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "없애다", "premise": "그녀는 욕실의 곰팡이를 없애고 있었다.", "hypothesis": "그녀는 욕실의 곰팡이를 다 없앴다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_036", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "자르다", "premise": "그는 종이 한 장을 자르고 있었다.", "hypothesis": "그는 종이 한 장을 전부 잘랐다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_037", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "썰다", "premise": "그녀는 양파 하나를 얇게 썰고 있었다.", "hypothesis": "그녀는 양파 하나를 얇게 다 썰었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_038", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "다지다", "premise": "그는 마늘을 다지고 있었다.", "hypothesis": "그는 마늘을 다 다졌다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_039", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "끓이다", "premise": "그녀는 물을 끓이고 있었다.", "hypothesis": "그녀는 물을 다 끓였다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_040", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "튀기다", "premise": "그는 감자를 튀기고 있었다.", "hypothesis": "그는 감자를 다 튀겼다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_041", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "굽다", "premise": "그녀는 고기를 다 익을 때까지 굽고 있었다.", "hypothesis": "그녀는 고기를 다 구웠다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_042", "verb_class": "Change_of_State", "secondary_class": "정도달성동사", "verb": "데우다", "premise": "그는 국 한 그릇을 데우고 있었다.", "hypothesis": "그는 국 한 그릇을 다 데웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_043", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "장식하다", "premise": "그녀는 케이크를 장식하고 있었다.", "hypothesis": "그녀는 케이크를 다 장식했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_044", "verb_class": "Change_of_State", "secondary_class": "완성동사", "verb": "들이다", "premise": "그들은 거실에 가구를 들이고 있었다.", "hypothesis": "그들은 거실에 가구를 다 들였다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_045", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "먹다", "premise": "그는 밥 한 그릇을 먹고 있었다.", "hypothesis": "그는 밥 한 그릇을 다 먹었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_046", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "마시다", "premise": "그녀는 물 한 잔을 마시고 있었다.", "hypothesis": "그녀는 물 한 잔을 다 마셨다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_047", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "읽다", "premise": "그는 책 한 권을 읽고 있었다.", "hypothesis": "그는 책 한 권을 다 읽었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_048", "verb_class": "Consumption", "secondary_class": "정도달성동사", "verb": "태우다", "premise": "그녀는 종이 한 장을 태우고 있었다.", "hypothesis": "그녀는 종이 한 장을 완전히 태웠다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_049", "verb_class": "Consumption", "secondary_class": "전형적 달성동사", "verb": "삼키다", "premise": "그는 약 한 알을 삼키고 있었다.", "hypothesis": "그는 약 한 알을 다 삼켰다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_050", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "소모하다", "premise": "차가 연료 1리터를 소모하고 있었다.", "hypothesis": "차가 연료 1리터를 전부 소모했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_051", "verb_class": "Consumption", "secondary_class": "전형적 달성동사", "verb": "부수다", "premise": "그들은 모래성 하나를 부수고 있었다.", "hypothesis": "그들은 모래성 하나를 모조리 부쉈다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_052", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "해결하다", "premise": "그는 수학 문제 하나를 해결하고 있었다.", "hypothesis": "그는 수학 문제 하나를 다 해결했다.", "ollama30b_zs": "False", "ollama30b_dap": "False"}, {"id": "C_053", "verb_class": "Consumption", "secondary_class": "완성동사", "verb": "벗기다", "premise": "그녀는 사과 하나의 껍질을 벗기고 있었다.", "hypothesis": "그녀는 사과 하나의 껍질을 다 벗겼다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_054", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "짓다", "premise": "그는 정자를 짓고 있었다.", "hypothesis": "그는 정자를 다 지었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_055", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "칠하다", "premise": "그녀는 벽 하나를 칠하고 있었다.", "hypothesis": "그녀는 벽 하나를 다 칠했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_056", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "새기다", "premise": "그는 목판에 글자를 새기고 있었다.", "hypothesis": "그는 목판에 글자를 다 새겼다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_057", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "쓰다", "premise": "그녀는 편지 한 통을 쓰고 있었다.", "hypothesis": "그녀는 편지 한 통을 다 썼다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_058", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "뜨개질하다", "premise": "그는 스웨터 한 벌을 뜨개질하고 있었다.", "hypothesis": "그는 스웨터 한 벌을 다 뜨개질했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_059", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "작곡하다", "premise": "그녀는 곡 하나를 작곡하고 있었다.", "hypothesis": "그녀는 곡 하나를 다 작곡했다.", "ollama30b_zs": "False", "ollama30b_dap": "False"}, {"id": "C_060", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "그리다", "premise": "그는 그림 한 점을 그리고 있었다.", "hypothesis": "그는 그림 한 점을 다 그렸다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_061", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "촬영하다", "premise": "그들은 영화 한 편을 촬영하고 있었다.", "hypothesis": "그들은 영화 한 편을 다 촬영했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_062", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "코딩하다", "premise": "그는 프로그램 하나를 코딩하고 있었다.", "hypothesis": "그는 프로그램 하나를 다 코딩했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_063", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "바느질하다", "premise": "그녀는 옷 한 벌을 바느질하고 있었다.", "hypothesis": "그녀는 옷 한 벌을 다 바느질했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_064", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "조각하다", "premise": "그는 조각상 하나를 조각하고 있었다.", "hypothesis": "그는 조각상 하나를 다 조각했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_065", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "짜다", "premise": "그녀는 천 한 필을 짜고 있었다.", "hypothesis": "그녀는 천 한 필을 다 짰다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_066", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "양조하다", "premise": "그는 맥주 한 통을 양조하고 있었다.", "hypothesis": "그는 맥주 한 통을 다 양조했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_067", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "위조하다", "premise": "그녀는 졸업장 하나를 위조하고 있었다.", "hypothesis": "그녀는 졸업장 하나를 다 위조했다.", "ollama30b_zs": "True", "ollama30b_dap": "False"}, {"id": "C_068", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "설계하다", "premise": "그는 건물 하나를 설계하고 있었다.", "hypothesis": "그는 건물 하나를 다 설계했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_069", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "증명하다", "premise": "그녀는 가설 하나를 증명하고 있었다.", "hypothesis": "그녀는 가설 하나를 다 증명했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_070", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "번역하다", "premise": "그는 소설 한 권을 번역하고 있었다.", "hypothesis": "그는 소설 한 권을 다 번역했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_071", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "건설하다", "premise": "그들은 다리 하나를 건설하고 있었다.", "hypothesis": "그들은 다리 하나를 다 건설했다.", "ollama30b_zs": "False", "ollama30b_dap": "False"}, {"id": "C_072", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "발명하다", "premise": "그녀는 기계 하나를 발명하고 있었다.", "hypothesis": "그녀는 기계 하나를 다 발명했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_073", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "생산하다", "premise": "공장이 자동차 100대를 생산하고 있었다.", "hypothesis": "공장이 자동차 100대를 생산했다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_074", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "세우다", "premise": "그는 여행 계획 하나를 세우고 있었다.", "hypothesis": "그는 여행 계획 하나를 다 세웠다.", "ollama30b_zs": "False", "ollama30b_dap": "False"}, {"id": "C_075", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "계산하다", "premise": "그녀는 총합을 계산하고 있었다.", "hypothesis": "그녀는 총합을 다 계산했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_076", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "측정하다", "premise": "그는 방의 넓이를 측정하고 있었다.", "hypothesis": "그는 방의 넓이를 다 측정했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_077", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "세다", "premise": "그녀는 동전 100개를 세고 있었다.", "hypothesis": "그녀는 동전 100개를 세었다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_078", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "받아쓰다", "premise": "그는 문장 하나를 받아쓰고 있었다.", "hypothesis": "그는 문장 하나를 다 받아썼다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_079", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "다운로드하다", "premise": "그녀는 파일 하나를 다운로드하고 있었다.", "hypothesis": "그녀는 파일 하나를 다 다운로드했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_080", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "업로드하다", "premise": "그는 영상 하나를 업로드하고 있었다.", "hypothesis": "그는 영상 하나를 다 업로드했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_081", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "인쇄하다", "premise": "그녀는 문서 한 부를 인쇄하고 있었다.", "hypothesis": "그녀는 문서 한 부를 다 인쇄했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_082", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "심다", "premise": "그는 나무 한 그루를 심고 있었다.", "hypothesis": "그는 나무 한 그루를 다 심었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_083", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "수확하다", "premise": "농부가 벼 한 마지기를 수확하고 있었다.", "hypothesis": "농부가 벼 한 마지기를 다 수확했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_084", "verb_class": "Creation", "secondary_class": "정도달성동사", "verb": "섞다", "premise": "그녀는 재료 세 가지를 섞고 있었다.", "hypothesis": "그녀는 재료 세 가지를 다 섞었다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_085", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "갖추다", "premise": "그들은 실험실에 장비를 갖추고 있었다.", "hypothesis": "그들은 실험실에 장비를 다 갖췄다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_086", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "설립하다", "premise": "그는 회사 하나를 설립하고 있었다.", "hypothesis": "그는 회사 하나를 다 설립했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_087", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "형성하다", "premise": "그녀는 새로운 습관을 형성하고 있었다.", "hypothesis": "그녀는 새로운 습관을 다 형성했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "False"}, {"id": "C_088", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "빚다", "premise": "그는 만두 100개를 빚고 있었다.", "hypothesis": "그는 만두 100개를 빚었다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_089", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "조립하다", "premise": "그녀는 가구 하나를 조립하고 있었다.", "hypothesis": "그녀는 가구 하나를 다 조립했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_090", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "모으다", "premise": "그는 동전 100개를 모으고 있었다.", "hypothesis": "그는 동전 100개를 모았다.", "ollama30b_zs": "True", "ollama30b_dap": "False"}, {"id": "C_091", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "수집하다", "premise": "그녀는 우표 50장을 수집하고 있었다.", "hypothesis": "그녀는 우표 50장을 수집했다.", "ollama30b_zs": "True", "ollama30b_dap": "Unknown"}, {"id": "C_092", "verb_class": "Creation", "secondary_class": "완성동사", "verb": "따다", "premise": "그는 사과 10개를 따고 있었다.", "hypothesis": "그는 사과 10개를 땄다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_093", "verb_class": "Motion_to_Goal", "secondary_class": "전형적 달성동사", "verb": "오르다", "premise": "그는 산 정상에 오르고 있었다.", "hypothesis": "그는 산 정상에 올랐다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_094", "verb_class": "Motion_to_Goal", "secondary_class": "완성동사", "verb": "건너다", "premise": "그녀는 강을 건너고 있었다.", "hypothesis": "그녀는 강을 건넜다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_095", "verb_class": "Motion_to_Goal", "secondary_class": "완성동사", "verb": "등산하다", "premise": "그는 정상까지 등산하고 있었다.", "hypothesis": "그는 정상까지 등산했다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_096", "verb_class": "Motion_to_Goal", "secondary_class": "완성동사", "verb": "날아가다", "premise": "비행기가 서울까지 날아가고 있었다.", "hypothesis": "비행기가 서울까지 날아갔다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_097", "verb_class": "Motion_to_Goal", "secondary_class": "완성동사", "verb": "배달하다", "premise": "그는 피자 한 판을 배달하고 있었다.", "hypothesis": "그는 피자 한 판을 배달했다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_098", "verb_class": "Motion_to_Goal", "secondary_class": "전형적 달성동사", "verb": "도착하다", "premise": "비행기가 공항에 도착하고 있었다.", "hypothesis": "비행기가 목적지 공항에 도착했다.", "ollama30b_zs": "Unknown", "ollama30b_dap": "Unknown"}, {"id": "C_099", "verb_class": "Motion_to_Goal", "secondary_class": "전형적 달성동사", "verb": "입장하다", "premise": "그녀는 경기장에 입장하고 있었다.", "hypothesis": "그녀는 경기장에 입장했다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}, {"id": "C_100", "verb_class": "Motion_to_Goal", "secondary_class": "전형적 달성동사", "verb": "도달하다", "premise": "그는 결승지점에 도달하고 있었다.", "hypothesis": "그는 결승지점에 도달했다.", "ollama30b_zs": "True", "ollama30b_dap": "True"}]''')
print(len(ITEMS_C), '문항')

In [ ]:
PROMPTS = {
    "zero-shot": {
        "system": """당신은 엄격한 논리학자입니다. 당신의 임무는 전제(Premise)가 주어졌을 때 가설(Hypothesis)이 반드시 참인지 판단하는 것입니다.
전제만으로 가설이 반드시 참이면 "참"을 출력하십시오.
가설이 전제와 모순되면 "거짓"을 출력하십시오.
가설이 참일 수도 있지만 전제에서 명시적으로 보장되지 않으면 "모름"을 출력하십시오.
상식적 추측을 사용하지 마십시오. 오직 제공된 텍스트만 사용하십시오.""",
        "user_format": """입력: 전제: {premise} 가설: {hypothesis}
다음 중 하나로만 답하십시오: "참", "거짓", 또는 "모름".""",
    },
    "dap": {
        "system": """당신은 엄격한 논리학자입니다. 당신의 임무는 전제(Premise)가 주어졌을 때 가설(Hypothesis)이 반드시 참인지 판단하는 것입니다.
전제만으로 가설이 반드시 참이면 "참"을 출력하십시오.
가설이 전제와 모순되면 "거짓"을 출력하십시오.
가설이 참일 수도 있지만 전제에서 명시적으로 보장되지 않으면 "모름"을 출력하십시오.

중요한 언어학적 규칙: 활동동사(activity)와 완수동사(accomplishment)의 차이에 주의하십시오.
- 완수동사/목표지향적 행위(예: "짓고 있었다", "쓰고 있었다")의 경우, 진행형이 완료를 의미하지 않습니다.
- 활동동사(예: "달리고 있었다", "노래하고 있었다")의 경우, 그 과정 자체가 행동이 일어났음을 의미합니다.
상식적 추측을 사용하지 마십시오. 오직 제공된 텍스트만 사용하십시오.""",
        "user_format": """입력: 전제: {premise} 가설: {hypothesis}
다음 중 하나로만 답하십시오: "참", "거짓", 또는 "모름".""",
    },
}
LABEL_MAP = {"참": "True", "거짓": "False", "모름": "Unknown"}
def parse_simple(raw_text):
    text = (raw_text or "").strip()
    found = [ko for ko in LABEL_MAP if ko in text]
    return {"label": LABEL_MAP[found[0]], "parse_ok": True} if len(found) == 1 else {"label": None, "parse_ok": False}

## 0-4. Logit lens 함수
각 층의 출력(잔차 흐름)에 모델의 마지막 정규화층과 출력층(lm_head)을 그대로 씌워, **'이 층에서 바로 답한다면' 다음 토큰 확률**을 계산한다. 위치는 프롬프트 마지막(첫 답 토큰을 예측하는 자리). 마지막 층 값이 실제 출력 확률과 같은지 자동으로 검산한다.

In [ ]:
def build_prompt(tok, strategy, it):
    p = PROMPTS[strategy]
    msgs = [{"role": "system", "content": p["system"]},
            {"role": "user", "content": p["user_format"].format(premise=it["premise"], hypothesis=it["hypothesis"])}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def label_token_ids(tok):
    ids = {}
    for ko, en in LABEL_MAP.items():
        t = tok.encode(ko, add_special_tokens=False)
        ids[en] = t[0]
        print(f'{ko}: 첫 토큰 {t[0]} {tok.decode([t[0]])!r}')
    assert len(set(ids.values())) == 3, '세 라벨의 첫 토큰이 겹칩니다'
    return ids

@torch.no_grad()
def lens_and_generate(model, tok, strategy, it, lab_ids):
    enc = tok(build_prompt(tok, strategy, it), return_tensors="pt").to(model.device)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states                      # (임베딩, 1층, ..., L층[최종 정규화 적용됨])
    head = model.get_output_embeddings(); norm = model.model.norm
    dev = head.weight.device
    ids = torch.tensor([lab_ids["True"], lab_ids["False"], lab_ids["Unknown"]], device=dev)
    layers = []
    L = len(hs) - 1
    for i in range(1, L + 1):
        h = hs[i][0, -1].to(dev)
        h = h if i == L else norm(h.unsqueeze(0))[0]
        logits = head(h.unsqueeze(0).to(head.weight.dtype))[0].float()
        pr = torch.softmax(logits, -1)
        top = int(pr.argmax())
        lp = pr[ids].tolist()
        layers.append({"layer": i, "p_true": lp[0], "p_false": lp[1], "p_unknown": lp[2], "top_token": tok.decode([top])})
    final = torch.softmax(out.logits[0, -1].float(), -1)[ids.to(out.logits.device)].tolist()
    assert abs(final[1] - layers[-1]["p_false"]) < 1e-3 and abs(final[2] - layers[-1]["p_unknown"]) < 1e-3, '마지막 층 검산 실패'
    gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(gen[0, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return raw, layers, L

In [ ]:
def run_model(tag, model, tok):
    path = os.path.join(OUT_DIR, f"{tag}_groupC_lens.jsonl")
    done = set()
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            if line.strip():
                r = json.loads(line); done.add((r["strategy"], r["id"]))
    lab_ids = label_token_ids(tok)
    t0 = time.time(); n = 0
    with open(path, "a", encoding="utf-8") as f:
        for s in STRATEGIES:
            for it in ITEMS_C:
                if (s, it["id"]) in done: continue
                raw, layers, L = lens_and_generate(model, tok, s, it, lab_ids)
                pr = parse_simple(raw)
                f.write(json.dumps(dict(model=tag, strategy=s, n_layers=L, raw_response=raw,
                        predicted_label=pr["label"], parse_ok=pr["parse_ok"], layers=layers, **it),
                        ensure_ascii=False) + "\n"); f.flush()
                n += 1
                if n % 20 == 0: print(f"  {tag}: {n}건 ({time.time()-t0:.0f}초)")
    print(f"{tag} 완료: 새로 {n}건, {time.time()-t0:.0f}초 → {path}")
    return path

## 1단계. Qwen3-4B (GPU 1개, fp16)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
if RUN_4B:
    tok4 = AutoTokenizer.from_pretrained(MODEL_4B, cache_dir=CACHE_DIR)
    m4 = AutoModelForCausalLM.from_pretrained(MODEL_4B, dtype=torch.float16, device_map={"": 0}, cache_dir=CACHE_DIR).eval()
    print('4B 층 수:', m4.config.num_hidden_layers)
    raw, layers, L = lens_and_generate(m4, tok4, "dap", ITEMS_C[0], label_token_ids(tok4))
    print('시험:', repr(raw), '| 마지막 3개 층 P(거짓/모름):', [(round(x["p_false"],3), round(x["p_unknown"],3)) for x in layers[-3:]])
    PATH_4B = run_model("qwen4b", m4, tok4)
    del m4; gc.collect(); torch.cuda.empty_cache()

## 2단계. Qwen3-30B 불러오기 (bnb NF4, GPU 2개에 나눠 올림)
원본 가중치(약 61GB)를 받은 뒤 불러오면서 4비트로 바꾼다. 다운로드 10~20분, 로드 10분 안팎. **전문가 층까지 4비트로 바뀌었는지 자동 확인**한다.

In [ ]:
if RUN_30B:
    from transformers import BitsAndBytesConfig
    import bitsandbytes as bnb
    src = MODEL_30B_LOCAL or MODEL_30B
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok30 = AutoTokenizer.from_pretrained(src, cache_dir=CACHE_DIR)
    m30 = AutoModelForCausalLM.from_pretrained(src, quantization_config=qcfg, device_map="auto",
                                               max_memory={0: "10GiB", 1: "13GiB", "cpu": "24GiB"},
                                               dtype=torch.float16, cache_dir=CACHE_DIR, low_cpu_mem_usage=True).eval()
    n4 = sum(1 for m in m30.modules() if isinstance(m, bnb.nn.Linear4bit))
    nlin = sum(1 for m in m30.modules() if isinstance(m, torch.nn.Linear) and not isinstance(m, bnb.nn.Linear4bit))
    print(f'4비트 층 {n4}개, 4비트가 아닌 Linear {nlin}개 (lm_head·router 등 소수여야 정상)')
    print('장치 배치:', {str(v) for v in m30.hf_device_map.values()})
    for i in range(torch.cuda.device_count()): print(f'GPU{i} 사용 {torch.cuda.memory_allocated(i)/1e9:.1f} GB')
    assert 'cpu' not in {str(v) for v in m30.hf_device_map.values()}, 'CPU로 넘어간 층이 있어 매우 느립니다 — 결과를 알려주세요'
    print('30B 층 수:', m30.config.num_hidden_layers)

## 3단계. Qwen3-30B 실행 (행동 + logit lens)

In [ ]:
if RUN_30B:
    raw, layers, L = lens_and_generate(m30, tok30, "dap", ITEMS_C[0], label_token_ids(tok30))
    print('시험:', repr(raw))
    PATH_30B = run_model("qwen30b_nf4", m30, tok30)

## 4단계. 사전 기준 판정 · 본실험(Ollama)과의 일치도

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import binomtest
def load(path):
    return pd.read_json(path, lines=True)
def judge(df, tag):
    ok = df[df.parse_ok]
    m = ok.groupby("strategy").predicted_label.value_counts(normalize=True).unstack(fill_value=0).reindex(columns=["Unknown","True","False"], fill_value=0)
    m.columns = ["ACC_C", "TBR_C", "FBR_C"]
    w = df.pivot(index="id", columns="strategy", values="predicted_label")
    b = int(((w["zero-shot"] != "False") & (w["dap"] == "False")).sum()); c = int(((w["zero-shot"] == "False") & (w["dap"] != "False")).sum())
    p = binomtest(b, b + c, 0.5, alternative="greater").pvalue if b + c else 1.0
    d = m.loc["dap", "FBR_C"] - m.loc["zero-shot", "FBR_C"]
    print(f"\n[{tag}] 파싱 성공 {df.parse_ok.mean():.0%}"); print(m.round(3))
    print(f"FBR_C 변화 {d:+.2f} | McNemar b={b}, c={c}, p={p:.3g} → {'재현' if d >= 0.15 and p < 0.05 else '미재현'}")
    return w
res = {}
if RUN_4B: res["4b"] = load(os.path.join(OUT_DIR, "qwen4b_groupC_lens.jsonl")); judge(res["4b"], "Qwen3-4B")
if RUN_30B:
    res["30b"] = load(os.path.join(OUT_DIR, "qwen30b_nf4_groupC_lens.jsonl")); w = judge(res["30b"], "Qwen3-30B NF4")
    ref = pd.DataFrame(ITEMS_C).set_index("id")
    for s, col in [("zero-shot", "ollama30b_zs"), ("dap", "ollama30b_dap")]:
        print(f"본실험(Ollama q4_K_M)과 문항별 일치 — {s}: {(w[s] == ref[col]).mean():.0%}")

## 5단계. 비교 그림: 층 깊이별 P(거짓)과 P(모름)
두 모델의 층 수(4B 36층, 30B 48층)가 달라 가로축은 **상대 깊이(0~1)**로 맞춘다. 실선 = DAP, 점선 = Zero-shot. 세 라벨 확률을 합이 1이 되도록 다시 나눈 값(라벨 간 상대 확률)을 쓴다.

In [ ]:
import matplotlib.pyplot as plt
def layer_frame(df):
    rows = []
    for _, r in df.iterrows():
        for x in r["layers"]:
            tot = x["p_true"] + x["p_false"] + x["p_unknown"] + 1e-12
            rows.append(dict(model=r["model"], strategy=r["strategy"], id=r["id"], depth=x["layer"] / r["n_layers"], layer=x["layer"],
                             false=x["p_false"]/tot, unknown=x["p_unknown"]/tot, true=x["p_true"]/tot, mass=tot,
                             final=r["predicted_label"], ollama_dap=r["ollama30b_dap"]))
    return pd.DataFrame(rows)
if res:
    LF = pd.concat([layer_frame(d) for d in res.values()])
    LF.to_csv(os.path.join(OUT_DIR, "layer_frame.csv"), index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, lab in zip(axes, ["false", "unknown"]):
        for (mdl, s), g in LF.groupby(["model", "strategy"]):
            mean = g.groupby("depth")[lab].mean()
            ax.plot(mean.index, mean.values, ls="-" if s == "dap" else "--", label=f"{mdl} / {s}")
        ax.set_title({"false": "P(False) - non-completion overconfidence", "unknown": "P(Unknown) - correct"}[lab]); ax.set_xlabel("relative depth (0 = input, 1 = output)")
    axes[0].set_ylabel("share among 3 labels (mean of 100 Group C items)"); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "lens_compare.png"), dpi=150); plt.show()
    print(LF.groupby(["model","strategy"]).apply(lambda g: g[g.depth > 0.5].mass.mean()).round(3).rename("상위 절반 층의 라벨 토큰 확률 합"))

## 6단계. 결과 내려받기
오른쪽 **Output** 패널에서 `lens_out` 폴더의 파일들을 내려받거나, 아래 셀로 zip 하나로 묶는다. 이 zip을 Claude에게 올려 주시면 층별 분기점 분석과 논문용 그림을 만든다.

In [ ]:
shutil.make_archive("/kaggle/working/lens_out", "zip", OUT_DIR)
print("/kaggle/working/lens_out.zip", round(os.path.getsize("/kaggle/working/lens_out.zip")/1e6, 1), "MB")